# DocuMind — RAG Pipeline Walkthrough

This notebook walks through *why* each stage of the pipeline works the way it does,
step by step, using the actual project code (not a toy reimplementation).

Sections:
1. Load & chunk a document
2. Embed chunks locally (no API key needed)
3. Store & query the vector index
4. Inspect retrieval quality
5. Generate a grounded answer with Claude
6. A small precision@k evaluation

Run this after `pip install -r requirements.txt` from the project root.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from app.ingest import load_text, chunk_text
from app import vectorstore, llm, rag
from app.config import settings

print("Embedding model:", settings.embedding_model)
print("Chunk size / overlap:", settings.chunk_size, "/", settings.chunk_overlap)

## 1. Load & chunk a document

Chunking matters more than people expect. Chunk too large and irrelevant text dilutes
the embedding, hurting retrieval precision. Chunk too small and you lose context the
LLM needs to answer well. This project chunks on paragraph boundaries up to a target
size, then stitches a small overlap between neighbors so an answer that straddles a
chunk boundary doesn't get cut in half.

In [ ]:
text = load_text("../data/sample_docs/sample.txt")
chunks = chunk_text(text)

print(f"Document length: {len(text)} chars -> {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c[:200], "...\n")

## 2. Embed chunks locally

Embeddings turn text into vectors where semantic closeness becomes geometric closeness.
We use `sentence-transformers/all-MiniLM-L6-v2` — small, fast, free, runs on CPU, and
good enough for most retrieval tasks. No Anthropic/OpenAI call happens at this stage.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)
vecs = model.encode(chunks[:2])
print("Embedding shape per chunk:", vecs.shape)
print("First 8 dims of chunk 0:", vecs[0][:8])

## 3. Store & query the vector index

This uses the same `vectorstore.py` module the FastAPI app calls. We reset the
collection here so this notebook is reproducible from a clean state.

In [ ]:
from app.ingest import ingest_directory

vectorstore.reset_collection()
summary = ingest_directory("../data/sample_docs")
print(summary)

## 4. Inspect retrieval quality

Before ever calling an LLM, it's worth looking at *what gets retrieved* for a query.
If the top-k chunks aren't actually relevant, no amount of prompt engineering will
fix the final answer — this is the most common failure point in real RAG systems.

In [ ]:
question = "How does DocuMind decide which text is relevant to a question?"
hits = vectorstore.query(question, top_k=3)

for h in hits:
    print(f"score={h['score']:.3f}  source={h['source']}")
    print(h['text'][:200], "...\n")

## 5. Generate a grounded answer

The retrieved chunks are inserted into a system-prompted call to Claude that's
instructed to answer *only* from context and to cite sources — this is what keeps
RAG answers from hallucinating facts not present in your documents.

Requires `ANTHROPIC_API_KEY` set in your `.env`.

In [ ]:
result = rag.answer_question(question)
print("ANSWER:\n", result["answer"])
print("\nSOURCES:", result["sources"])

## 6. A minimal precision@k evaluation

A real evaluation harness is what separates a demo from a project you can defend in
an interview. Here's a tiny example: for a handful of question/expected-source pairs,
check whether the correct source shows up in the top-k retrieved chunks.

Extend this with a labeled question set from your own documents for a real eval.

In [ ]:
eval_set = [
    {"question": "What embedding model does DocuMind use?", "expected_source": "sample.txt"},
    {"question": "Why use a vector database instead of retraining a model?", "expected_source": "sample.txt"},
]

correct = 0
for case in eval_set:
    hits = vectorstore.query(case["question"], top_k=3)
    retrieved_sources = {h["source"] for h in hits}
    hit = case["expected_source"] in retrieved_sources
    correct += hit
    print(f"[{'HIT' if hit else 'MISS'}] {case['question']}")

print(f"\nPrecision@3 (source-level): {correct}/{len(eval_set)}")

## Takeaways

- Retrieval quality is the bottleneck, not the LLM. Inspect it directly before blaming generation.
- Grounding the prompt with explicit "answer only from context" instructions materially reduces hallucination.
- A tiny eval harness like the one above is easy to extend into a real regression test suite — worth building out further before calling a RAG project "done".